# LSTM Time Series - Stock Price Prediction
## Part 3 - Model Training
In this notebook, we import the scaled dataset files, prepare them in a format suitable for LSTM modeling, and proceed to train the LSTM model.

> **INPUT**: Scaled dataset files for training, validation, and testing periods, as processed in the preceding phase. <br/>
> **OUTPUT**: Trained LSTM model and analysis of its performance.

### 1. INITIALIZATION

In [1]:
import sys
import optuna
optuna.__version__

sys.path.append('/Users/rifatordulu/Developer/lstm-stock-price-prediction/custom_objects')
%run "../helpers/data_manipulator.py"
%run "../helpers/yfinance_data_fetcher.py"
%run "../custom_objects/custom_objects.py"

from custom_objects import register_custom_objects, precision_with_threshold, recall_with_threshold, focal_loss, true_positives, all_positives, recall_mul_prediction, f1_score_metric, cubic_loss

# Register globally
register_custom_objects()

In [2]:
NUM_EPOCH = 200
SEQUENCE_SIZE = 65
# SEQUENCE_SIZE = 53
# SEQUENCE_SIZE = 21
REDUCE_LR_PATIENCE = 6
STOP_L_PATIENCE = 16
BATCH_SIZE = 128
RELOAD_FROM_NUMPY_FILE = False
NORMALIZE_Y = True
use_pre_calc_sigma = True
use_extra_dense_layers = False
do_stock_by_stock = True
do_shuffle_for_train = False

if is_binary_prediction:
    STARTING_LR = 0.0001
    # LOSS_FUNCTION = "mae"
    LOSS_FUNCTION = "weighted_bce"
    early_stop_delta = 0.0001
    save_best_metric = "val_AUC"
    save_best_mode = "max"
    monitor = "val_loss"
    mode = "min"
else:
    STARTING_LR = 0.0003
    LOSS_FUNCTION = "rifat"
    early_stop_delta = 0.00001
    save_best_metric = "val_loss"
    save_best_mode = "min"
    monitor = "val_loss"
    mode = "min"

SPEC_FILE_TAG = "last_trained_model"
# "PFE", "ABBV", "COST", "TMO", "DIS", "CSCO", "MCD", 
# stocks = ["DHR", "NKE", "LIN", "ACN", "ABT", "CVX", "NEE", "TXN", "MDT", "CRM", "ORCL", "UPS", "PM", "AMD", "HON", "MS", "UNP", "IBM", "INTC", "AMGN", "QCOM", "SPGI", "RTX", "LOW", "GS", "CAT", "NOW", "BLK", "GE", "LMT", "SCHW", "ADBE", "ELV", "PLD", "BKNG", "T", "DE", "SBUX", "ISRG", "MDLZ", "MO", "ADP", "SYK", "ZTS", "CB", "CI", "SO", "MMC", "GILD", "USB", "PGR", "FIS", "ADI"]
# stocks = all_stocks
target_options = [
    # "Next-Day-Close-To-Next-Day-Open-Ratio"
    # , 
                  "Next-Day-High-To-Next-Day-Open-Ratio"
]
# stocks = get_stocks_with_cluster(4)
# stocks = ["META"]
# stocks = ["NFLX"]
# stocks = ["MSCI"]
# stocks = ["GOOGL"]
stocks = ["SPOT"]
# stocks = ["QQQ"]
# stocks = ["GOOGL", "NFLX", "META", "SPOT"]
# stocks = limited_stocks
# stocks = my_stocks
# stocks = [
#     # "TSLA", "NVDA", "AAPL", "MSTR", "MSFT"
#     #       , "AVGO", "PLTR", "META", 
#           # "AMZN", "UNH"
#           # , "JPM", "AMD", "RGTI", "BAC", "NFLX", "GOOG"
#           # , 
#     # "IONQ", "LLY", "UBER", "CRM", "MU", "T", "F"
#     #       , "PFE", 
#     # "XOM", "WMT", "DIS", "KO", "INTC", "CSCO"
#     #       , "BA", 
#     # "V", "JNJ", "WFC", "BABA", "SPY", "QQQ", "DIA"
#     #       , 
#     # "DASH", "SPOT",  "ABNB"
#     # ,"GLD", "SLV", "TLT", "IWM", "GME", 
#     "AMC", "BBBY", "NIO"
#           , "LCID", "RIVN", "NKLA", "PLUG", "SPCE", "SQ", "PYPL"
#           , "SHOP", "ZM", "ROKU", "SNAP", "TWTR", "PINS"
#           , "UBER", "LYFT", "PTON", "DOCU", "ZM"
#           , "CRWD", "NET", "DDOG", "MDB", "ZS", "OKTA", "TEAM", "WORK"
#           , "FSLY", "U", "RBLX", "ATVI", "EA", "TTWO", "SONY"
#           , "NTDOY", "TCEHY", "BILI", "IQ", "HUYA", "DOYU", "YY"
#           , "JD", "PDD", "BIDU", "TAL", "EDU", "GOTU", "YMM"
#           , "DIDI", "BABA", "TME", "IQ", "HUYA", "DOYU"]
# stocks = stocks + ["GOOGL"]
# stocks = get_stocks_with_cluster(1)

if len(stocks) > 50:
    L2_RATE = 0.01
elif len(stocks) > 10:
    L2_RATE = 0.001
else:
    L2_RATE = 0.01

In [3]:
# Import necessary libraries and modules
from tensorflow.keras.models import Model, Sequential
from tensorflow.keras.layers import Input, LSTM, Dropout, Dense, Conv1D, BatchNormalization, PReLU, LayerNormalization, Reshape, Lambda, Flatten
from tensorflow.keras.regularizers import l2
from tensorflow.keras.callbacks import ReduceLROnPlateau, EarlyStopping, ModelCheckpoint
from tensorflow.keras.models import load_model
from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras.optimizers import Adam
from sklearn.utils.class_weight import compute_class_weight
from sklearn.model_selection import train_test_split
from sklearn.model_selection import train_test_split
import pandas as pd
from sklearn.preprocessing import LabelEncoder
import pickle
import math
import tensorflow.keras.backend as K
import os
import logging
import absl.logging
import joblib
import matplotlib.dates as mdates
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from keras_tuner import HyperParameters
from keras_tuner.tuners import RandomSearch
import logging
import datetime

os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'
tf.get_logger().setLevel('ERROR')
# Suppress absl.logging (used internally by TensorFlow)
logging.getLogger('absl').setLevel(logging.ERROR)


### 5. TRAINING LSTM MODEL

In [4]:

def construct_data(seq_len, stocks, underlying_target):
    data_frame, real_target = construct_values_for_model(sequence_size = seq_len,
                                                        ticker_symbols=stocks,
                                                        underlying_target=underlying_target,
                                                        use_for_last_day_prediction=False,
                                                        verbose=False,
                                                        refresh=False,
                                                        data_interval="5y",
                                                        find_sectors=True)
    two_sigma_cap_per_ticker = None
    if not use_pre_calc_sigma:
        two_sigma_cap_per_ticker = pd.DataFrame()
        for ticker in data_frame["Orig_Ticker"].unique(): 
            mean_value = data_frame.loc[data_frame["Orig_Ticker"] == ticker, "y-value-original"].mean()
            std_dev_value = data_frame.loc[data_frame["Orig_Ticker"] == ticker, "y-value-original"].std()
             # Filter out the 3-sigma range
            upper_bound = mean_value + 3 * std_dev_value
            print(f"size before sigma app: {len(data_frame)} for ticker: {ticker}")
            data_frame = data_frame.drop(data_frame[((data_frame["Orig_Ticker"] == ticker) & (data_frame["y-value-original"] > upper_bound))].index)
            print(f"size after sigma app: {len(data_frame)} for ticker: {ticker}")
        
            new_row = pd.DataFrame({"Orig_Ticker": [ticker], "2-sigma": [mean_value + 3.0 * std_dev_value]})
            two_sigma_cap_per_ticker = pd.concat([two_sigma_cap_per_ticker, new_row], ignore_index=True)

    if do_shuffle_for_train:
        train_data, validation_data, test_data = split_data(data_frame,
                                                        train_cut_date = "2024-11-30",
                                                        validate_cut_date = "2024-11-30",
                                                        test_end_date = "2024-12-30",
                                                           # train_start_date = "2023-01-01"
                                                           )
        print(f"1. Train data length: {len(train_data)}")
        print(f"1. Validation data length: {len(validation_data)}")
        train_data, validation_data = train_test_split(train_data,
                                                       test_size=0.2,
                                                       random_state=12,
                                                       shuffle=True)
        print(f"2. Train data length: {len(train_data)}")
        print(f"2. Validation data length: {len(validation_data)}")
    else:
        train_data, validation_data, test_data = split_data(data_frame,
                                                            train_cut_date = "2024-10-01",
                                                            validate_cut_date = "2024-12-31",
                                                            test_end_date = "2024-12-31",
                                                               # train_start_date = "2023-01-01"
                                                               )
    
    print(train_data["y-value"])
    print(train_data["y-value-original"])
    # BElow verifying the arr conversions work properly. check the values...
    print(train_data.iloc[0]["LstmData"].iloc[0])
    print(np.array(train_data["LstmData"].to_list())[0][0])
    
    
    print(f"train data shape:{train_data.shape} and train data columns: {train_data.columns}")
    print(f"validation_data shape:{validation_data.shape} and validation_data columns: {validation_data.columns}")
    print(f"test_data shape:{test_data.shape} and test_data columns: {test_data.columns}")

    return train_data, validation_data, test_data, two_sigma_cap_per_ticker, real_target

#### Building LSTM Model

In [5]:
from tensorflow.keras.utils import plot_model

def create_model(train_data, network_type, seq_len):

    ### MODEL 1
    from tensorflow.keras.layers import Concatenate, ZeroPadding1D, Cropping1D, Embedding, RepeatVector, Input, Reshape, Conv1D, PReLU, LSTM, Dropout, Dense, Flatten
    
    input_data = train_data.iloc[0]["LstmData"]
    print(input_data.shape)
    
    input_layer = tf.keras.layers.Input(shape=(input_data.shape[0], input_data.shape[1]), name="lstm-values")
    stock_id_input = tf.keras.layers.Input(shape=(1,), name="stock_id")  # Stock ID input
    sector_input = tf.keras.layers.Input(shape=(1,), name="sector_id")  # Sector ID input
    prev_day_input = tf.keras.layers.Input(shape=(len(lstm_features),), name="prev-day-values")
    
    
    # Stock Embedding Layer
    num_of_unique_stocks = train_data["Ticker"].nunique()  # Adjust based on the number of unique stocks
    num_of_unique_sectors = train_data["Sector"].nunique()  # Adjust based on the number of unique stocks
    
    # Roll to the nearest higher integer
    stock_embedding_dim = 512 #math.ceil(math.sqrt(num_of_unique_stocks)) * 2
    sector_embedding_dim = 128 #math.ceil(math.sqrt(num_of_unique_sectors)) * 2
    
    stock_embedding = tf.keras.layers.Embedding(input_dim=num_of_unique_stocks, output_dim=stock_embedding_dim, name="stock_embedding")(stock_id_input)
    sector_embedding = tf.keras.layers.Embedding(input_dim=num_of_unique_sectors, output_dim=sector_embedding_dim, name="sector_embedding")(sector_input)
    
    # Reshape to remove the sequence length dimension
    stock_embedding = tf.keras.layers.Reshape((stock_embedding_dim,))(stock_embedding)
    sector_embedding = tf.keras.layers.Reshape((sector_embedding_dim,))(sector_embedding)

    if network_type == "cnn" or network_type == "lstm+cnn":
        cnn = tf.keras.layers.Conv1D(256,3, kernel_regularizer=l2(L2_RATE))(input_layer)
        cnn = tf.keras.layers.PReLU(shared_axes=[1], alpha_initializer=tf.initializers.constant(0))(cnn)
        # cnn = tf.keras.layers.AveragePooling1D(pool_size=2)(cnn)
        cnn = tf.keras.layers.Conv1D(256,3, strides=2, kernel_regularizer=l2(L2_RATE))(cnn)
        cnn = tf.keras.layers.PReLU(shared_axes=[1], alpha_initializer=tf.initializers.constant(0))(cnn)
        # cnn = tf.keras.layers.AveragePooling1D(pool_size=2)(cnn)
        cnn = tf.keras.layers.Conv1D(512,3, strides=2, kernel_regularizer=l2(L2_RATE))(cnn)
        cnn = tf.keras.layers.PReLU(shared_axes=[1], alpha_initializer=tf.initializers.constant(0))(cnn)
        # cnn = tf.keras.layers.AveragePooling1D(pool_size=2)(cnn)
        cnn = tf.keras.layers.Conv1D(512,3, strides=2, kernel_regularizer=l2(L2_RATE))(cnn)
        cnn = tf.keras.layers.PReLU(shared_axes=[1], alpha_initializer=tf.initializers.constant(0))(cnn)
        # cnn = tf.keras.layers.MaxPooling1D(pool_size=2)(cnn)
        cnn = tf.keras.layers.Conv1D(1024,3, strides=2,kernel_regularizer=l2(L2_RATE))(cnn)
        cnn = tf.keras.layers.PReLU(shared_axes=[1], alpha_initializer=tf.initializers.constant(0))(cnn)
        # cnn = tf.keras.layers.MaxPooling1D(pool_size=2)(cnn)
        cnn = tf.keras.layers.Conv1D(1024,3, strides=2,kernel_regularizer=l2(L2_RATE))(cnn)
        cnn = tf.keras.layers.PReLU(shared_axes=[1], alpha_initializer=tf.initializers.constant(0))(cnn)
        
        cnn = tf.keras.layers.Flatten()(cnn)
        
        # cnn = tf.keras.layers.Dense(800)(cnn)
        # cnn = tf.keras.layers.PReLU(shared_axes=[1], alpha_initializer=tf.initializers.constant(0.2))(cnn)
        # cnn = tf.keras.layers.Dense(800)(cnn)
        # cnn = tf.keras.layers.PReLU(shared_axes=[1], alpha_initializer=tf.initializers.constant(0.2))(cnn)
        # cnn = tf.keras.layers.Conv1D(512,3, kernel_regularizer=l2(0.00001))(cnn)
        # cnn = tf.keras.layers.PReLU(shared_axes=[1], alpha_initializer=tf.initializers.constant(0.25))(cnn)
        # cnn = tf.keras.layers.Conv1D(512,3, kernel_regularizer=l2(0.00001))(cnn)
        # cnn = tf.keras.layers.PReLU(shared_axes=[1], alpha_initializer=tf.initializers.constant(0.25))(cnn)
        # cnn = tf.keras.layers.Conv1D(1024,3, kernel_regularizer=l2(0.00001))(cnn)
        # cnn = tf.keras.layers.PReLU(shared_axes=[1], alpha_initializer=tf.initializers.constant(0.25))(cnn)
    else:
        cnn = input_layer

    if use_extra_dense_layers:
        extra = tf.keras.layers.Dense(200)(prev_day_input)
        extra = tf.keras.layers.PReLU(shared_axes=[1], alpha_initializer=tf.initializers.constant(0.2))(extra)
        extra = tf.keras.layers.Dense(200)(extra)
        extra = tf.keras.layers.PReLU(shared_axes=[1], alpha_initializer=tf.initializers.constant(0.2))(extra)
        extra = tf.keras.layers.Dense(100)(extra)
        extra = tf.keras.layers.PReLU(shared_axes=[1], alpha_initializer=tf.initializers.constant(0.2))(extra)
        cnn = Concatenate(axis=-1)([cnn, extra])
    
    # Pass embeddings through a dense layer
    stock_embedding = tf.keras.layers.Dense(512, activation='relu', name="stock_dense")(stock_embedding)
    sector_embedding = tf.keras.layers.Dense(256, activation='relu', name="sector_dense")(sector_embedding)
    
    # stock_embedding = tf.keras.layers.Dropout(0.2)(stock_embedding)
    # sector_embedding = tf.keras.layers.Dropout(0.2)(sector_embedding)
    
    # stock_embedding = tf.keras.layers.RepeatVector(cnn.shape[1])(stock_embedding)  # Match timesteps with input data
    cnn = Concatenate(axis=-1)([cnn, stock_embedding])
    
    # sector_embedding = tf.keras.layers.RepeatVector(cnn.shape[1])(sector_embedding)  # Match timesteps with input data
    cnn = Concatenate(axis=-1)([cnn, sector_embedding])

    lstm = cnn
    
    if network_type == "lstm" or network_type == "lstm+cnn":
        lstm = tf.keras.layers.LSTM(units = 100, return_sequences = True)(lstm)
        lstm = tf.keras.layers.Dropout(rate = 0.2)(lstm)
        lstm = tf.keras.layers.LSTM(units = 100, return_sequences = True)(lstm)
        lstm = tf.keras.layers.Dropout(rate = 0.2)(lstm)
        lstm = tf.keras.layers.LSTM(units = 100, return_sequences = True)(lstm)
        lstm = tf.keras.layers.Dropout(rate = 0.2)(lstm)
        lstm = tf.keras.layers.LSTM(units = 100)(lstm)
        lstm = tf.keras.layers.Dropout(rate = 0.2)(lstm)
    else:
        # Below is needed so we take away all the dimensions from the CNN (channels)
        lstm = tf.keras.layers.Dense(2000)(lstm)
        lstm = tf.keras.layers.PReLU(shared_axes=[1], alpha_initializer=tf.initializers.constant(0.2))(lstm)
        lstm = tf.keras.layers.Dropout(rate = 0.1)(lstm)
        # lstm = tf.keras.layers.Dense(4000)(lstm)
        # lstm = tf.keras.layers.PReLU(shared_axes=[1], alpha_initializer=tf.initializers.constant(0.2))(lstm)
        # lstm = tf.keras.layers.Dropout(rate = 0.1)(lstm)
        lstm = tf.keras.layers.Dense(400)(lstm)
        lstm = tf.keras.layers.PReLU(shared_axes=[1], alpha_initializer=tf.initializers.constant(0.2))(lstm)
        lstm = tf.keras.layers.Dropout(rate = 0.1)(lstm)
        # lstm = tf.keras.layers.Dense(1000)(lstm)
        # lstm = tf.keras.layers.PReLU(shared_axes=[1], alpha_initializer=tf.initializers.constant(0.2))(lstm)
        # lstm = tf.keras.layers.Dropout(rate = 0.1)(lstm)
        # lstm = tf.keras.layers.Dense(250)(lstm)
        # lstm = tf.keras.layers.PReLU(shared_axes=[1], alpha_initializer=tf.initializers.constant(0.2))(lstm)
        # lstm = tf.keras.layers.Dropout(rate = 0.1)(lstm)
        lstm = tf.keras.layers.Dense(100)(lstm)
        lstm = tf.keras.layers.PReLU(shared_axes=[1], alpha_initializer=tf.initializers.constant(0.2))(lstm)
        lstm = tf.keras.layers.Dropout(rate = 0.1)(lstm)
        # lstm = tf.keras.layers.Dense(200)(lstm)
        # lstm = tf.keras.layers.PReLU(shared_axes=[1], alpha_initializer=tf.initializers.constant(0.2))(lstm)
        # lstm = tf.keras.layers.Dense(100)(lstm)
        # lstm = tf.keras.layers.PReLU(shared_axes=[1], alpha_initializer=tf.initializers.constant(0.2))(lstm)
        # lstm = tf.keras.layers.Dense(50)(lstm)
        # lstm = tf.keras.layers.PReLU(shared_axes=[1], alpha_initializer=tf.initializers.constant(0.2))(lstm)
        # lstm = tf.keras.layers.Dense(20)(lstm)
        # lstm = tf.keras.layers.PReLU(shared_axes=[1], alpha_initializer=tf.initializers.constant(0.2))(lstm)
        # lstm = tf.keras.layers.Dense(10)(lstm)
        # lstm = tf.keras.layers.PReLU(shared_axes=[1], alpha_initializer=tf.initializers.constant(0.2))(lstm)
        # lstm = tf.keras.layers.Dense(4)(lstm)
        # lstm = tf.keras.layers.PReLU(shared_axes=[1], alpha_initializer=tf.initializers.constant(0.2))(lstm)
        # lstm = tf.keras.layers.Dense(2, activation='relu')(lstm)
        lstm = tf.keras.layers.Flatten()(lstm)
        
    # lstm = tf.keras.layers.Dense(1)(lstm)
    lstm = tf.keras.layers.Dense(1, activation='sigmoid')(lstm)
    regressor = tf.keras.models.Model([input_layer, stock_id_input, sector_input, prev_day_input],lstm)
    regressor.summary()
    plot_model(regressor, to_file='regressor_tree.png', show_shapes=True, show_layer_names=True)
    return regressor

### Compile the model

In [6]:
def calculate_loss(weight_ratio = 0.0):
    loss = None
    loss_name = None
    
    if LOSS_FUNCTION == "mse":
        loss = "mse"
        loss_name = "mse"
    elif LOSS_FUNCTION == "focal":
        loss = focal_loss(alpha=0.25, gamma=2.0)
        loss_name = "focal_loss"
    elif LOSS_FUNCTION == "mae":
        loss = "mae"
        loss_name = "mae"
    elif LOSS_FUNCTION == "cubic":
        loss = cubic_loss(power=3)
        loss_name="cubic"
    elif LOSS_FUNCTION == "square":
        loss = cubic_loss(power=0.5)
        loss_name="square"
    elif LOSS_FUNCTION == "weighted_bce":
        loss = weighted_bce(ratio=weight_ratio)
        loss_name="weighted_bce"
    elif LOSS_FUNCTION == "rifat":
        loss = rifat_loss(ratio=weight_ratio)
        loss_name="rifat"
    else:
        loss = "binary_crossentropy"
        loss_name = "binary_crossentropy"
    return loss, loss_name

### START THE TRAINING

In [7]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

# Define chart colors
train_actual_color = "cornflowerblue"
validate_actual_color = "orange"
test_actual_color = "green"
train_predicted_color = "lightblue"
validate_predicted_color = "peru"
test_predicted_color = "limegreen"

def calculate_for(validation_data, folder, val=0, two_sigma_cap_per_ticker=None, min_threshold=0.01, max_threshold=0.03):


    if not is_binary_prediction:
        if use_pre_calc_sigma:
            validation_data['y-predict-binary'] = validation_data['y-predict-original'] > min_threshold #validation_data["Target-20D-Mean"].apply(lambda x: min(max(x, min_threshold), max_threshold))
            validation_data['y-value-binary'] = validation_data['y-value-original'] > min_threshold #validation_data["Target-20D-Mean"].apply(lambda x: min(max(x, min_threshold), max_threshold))
        elif two_sigma_cap_per_ticker is not None:
            for ticker in validation_data["Orig_Ticker"].unique():
                two_sigma = two_sigma_cap_per_ticker[two_sigma_cap_per_ticker["Orig_Ticker"] == ticker].iloc[0]["2-sigma"]
                validation_data.loc[validation_data["Orig_Ticker"] == ticker, 'y-predict-binary'] = validation_data['y-predict-original'].apply(lambda x: 1 if x > two_sigma else 0)
                validation_data.loc[validation_data["Orig_Ticker"] == ticker, 'y-value-binary'] = validation_data['y-value-original'].apply(lambda x: 1 if x > two_sigma else 0)
    else:
        validation_data.loc[:,'y-predict-binary'] = validation_data['y-predict-original'].apply(lambda x: 1 if x > val else 0)
        validation_data.loc[:,'y-value-binary'] = validation_data['y-value-original'].apply(lambda x: 1 if x > val else 0)
        

    validation_data.loc[:,'true-positive'] = (validation_data["y-predict-binary"] == 1.0) & (validation_data["y-value-binary"] == 1.0)
    validation_data.loc[:,'false-positive'] = (validation_data["y-predict-binary"] == 1.0) & (validation_data["y-value-binary"] == 0.0)
    validation_data.loc[:,'false-negative'] = (validation_data["y-predict-binary"] == 0.0) & (validation_data["y-value-binary"] == 1.0)

    tps = validation_data['true-positive'].sum()
    fps = validation_data['false-positive'].sum()
    fns = validation_data['false-negative'].sum()
    real_trues = validation_data['y-value-binary'].sum()
    all_values = len(validation_data)
    
    print(f"Number of true positives for the given ticker: {tps}")
    print(f"Number of false positives for the given ticker: {fps}")
    print(f"Precision score: {tps/ (tps+fps)}")
    print(f"Recall score: {tps/ (tps+fns)}")
    print(f"Random guess score: {real_trues/ all_values}")

    
    all_per_ticker = validation_data.groupby('Orig_Ticker').size()
    tp_per_ticker = validation_data.groupby('Orig_Ticker')['true-positive'].sum()
    fp_per_ticker = validation_data.groupby('Orig_Ticker')['false-positive'].sum()
    fn_per_ticker = validation_data.groupby('Orig_Ticker')['false-negative'].sum()
    precision_per_ticker = tp_per_ticker / (tp_per_ticker + fp_per_ticker)
    recall_per_ticker = tp_per_ticker / (tp_per_ticker + fn_per_ticker)
    rand_guess_per_ticker = (tp_per_ticker + fn_per_ticker) / all_per_ticker
    
    print(f"true-positive per ticker: {tp_per_ticker}")
    print(f"false-positive per ticker: {fp_per_ticker}")
    print(f"false-negative per ticker: {fn_per_ticker}")
    print(f"precision per ticker: {precision_per_ticker}")
    print(f"recall per ticker: {recall_per_ticker}")
    print(f"rand guess per ticker: {rand_guess_per_ticker}")

    if is_binary_prediction:
        print_true = validation_data[validation_data['true-positive'] == 1.0][["Date", "Orig_Ticker", "y-predict-original", "y-value-original", "Target-20D-Mean", "Target-20D-Sigma"]]
        print_false = validation_data[validation_data['false-positive'] == 1.0][["Date", "Orig_Ticker", "y-predict-original", "y-value-original", "Target-20D-Mean", "Target-20D-Sigma"]]
        
        print(f"Some true positives")
        print(print_true.sample(n=min(5, len(print_true)), random_state=42))
    
        print(f"Some false positives")
        print(print_false.sample(n=min(5, len(print_false)), random_state=42))

    score = (recall_per_ticker > 0.0) * (precision_per_ticker / rand_guess_per_ticker)
    
    gain = 0.02 * tps
    loss = 0.01 * fps
    
    print(f"gain is: {gain} and loss is {loss}")
    print(f"Effective gain is: {gain - loss}")
    print(f"Score per ticker is: {score}")
    print(f"Total score is: {score.sum()}")

    
    ticker_to_show = validation_data.iloc[0]["Orig_Ticker"]
    ticker_data = validation_data[validation_data['Orig_Ticker'] == ticker_to_show]

    if is_binary_prediction:
        # Plot actual and predicted price
        plt.figure(figsize=(18,6))
        plt.plot(ticker_data['y-predict-binary'], label=f"{ticker_to_show} Prediction Values", color=train_predicted_color, marker='o', linestyle='')
        plt.plot(ticker_data['y-value-binary'], label=f"{ticker_to_show} Real Values", color=train_predicted_color, marker='x', linestyle='')
        plt.title("Test results")
        plt.xlabel("Time")
        plt.ylabel("Binary Prediction - True/False positives")
        plt.xticks(rotation=45)
        plt.legend()
        plt.grid(color="lightgray")
        plt.savefig(folder + "Inverse-normalized-real-vs-predicted_")
    else:
        plt.figure(figsize=(18,6))
        plt.plot(ticker_data['y-predict-original'], label=f"{ticker_to_show} Prediction Values", color=train_predicted_color, marker='o', linestyle='')
        plt.plot(ticker_data['y-value-original'], label=f"{ticker_to_show} Real Values", color=train_predicted_color, marker='x', linestyle='')
        plt.title("Test results")
        plt.xlabel("Time")
        plt.ylabel("Outright Prediction")
        plt.xticks(rotation=45)
        plt.legend()
        plt.grid(color="lightgray")
        plt.savefig(folder + "outright-prediction-normalized-real-vs-predicted_")

    return score.sum()
    

In [8]:
def print_and_save_for(data, title, folder, label):
    print(f"\n\nprint_and_save_for")
    ticker = data['Orig_Ticker'].unique()[0]
    data = data[data['Orig_Ticker'] == ticker]

    print(f"ticker data: {ticker}\n\n")
    print(f"Here is the data: {data} \n\n\n")
    
    plt.figure(figsize=(18,6))
    plt.plot(data['Date'], data['y-predict-original'], label=f"{ticker} Prediction Values", color=train_predicted_color, marker='o', linestyle='')
    plt.plot(data['Date'], data['y-value-original'], label=f"{ticker} Real Values", color=train_predicted_color, marker='x', linestyle='')
    plt.title(f"{title} results")
    plt.xlabel("Time")
    plt.ylabel("Prediction")
    plt.xticks(rotation=45)
    plt.legend()
    plt.grid(color="lightgray")
    plt.savefig(folder + "-Real-vs-prediction-values-" + title)
    plt.show()
    print(f"DONE PLOTTING \n\n")

In [9]:
from sklearn.metrics import precision_recall_curve
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import sklearn

def plot_precision_recall_curve(data_frame, folder, name="All Tickers"):
    precision, recall, thresholds = precision_recall_curve(data_frame["y-value"].to_list(), data_frame["y-predict"].to_list())
    
    # Extend thresholds to include 1.0 for completeness
    thresholds = np.append(thresholds, 1.0)
    
    # Plot precision and recall against thresholds
    plt.figure(figsize=(10, 6))
    plt.plot(thresholds, precision, label='Precision', color='blue')
    plt.plot(thresholds, recall, label='Recall', color='red')
    
    # Highlight the threshold where recall >= 0.02
    valid_indices = [i for i, r in enumerate(recall) if r > 0.0]
    highlight_thresholds = thresholds[valid_indices]
    highlight_precision = precision[valid_indices]
    highlight_recall = recall[valid_indices]
    plt.scatter(highlight_thresholds, highlight_precision, color='blue', label='Valid Precision Points', s=10)
    plt.scatter(highlight_thresholds, highlight_recall, color='red', label='Valid Recall Points', s=10)
    
    # Add labels and legend
    plt.xlabel('Threshold')
    plt.ylabel('Precision / Recall')
    plt.title(name + "- Precision-Recall vs. Threshold")
    plt.legend()
    plt.grid()
    plt.savefig(folder + "-Precision-Recall-Curve-")
    plt.show()

colors = plt.rcParams['axes.prop_cycle'].by_key()['color']

def plot_roc(name, labels, predictions, folder, **kwargs):
    fp, tp, _ = sklearn.metrics.roc_curve(labels, predictions)

    print(fp)
    print(tp)
    plt.plot(100*fp, 100*tp, label=name, linewidth=2, **kwargs)
    plt.xlabel('False positives [%]')
    plt.ylabel('True positives [%]')
    plt.xlim([-0.5,100])
    plt.ylim([0,100])
    plt.grid(True)
    ax = plt.gca()
    ax.set_aspect('equal')

def plot_roc_and_PR_curvers_for(train_data, validation_data, folder):
    if is_binary_prediction:
        plot_precision_recall_curve(validation_data, folder)
        for ticker in validation_data["Orig_Ticker"].unique():    
            plot_precision_recall_curve(validation_data.loc[validation_data["Orig_Ticker"] == ticker], folder, name=ticker)
            
        plt.figure(figsize=(10, 6))
        plot_roc("Train Baseline", train_data["y-value"], train_data["y-predict"], folder, color=colors[0])
        plot_roc("Test Baseline", validation_data["y-value-original"], validation_data["y-predict-original"], folder, color=colors[0], linestyle='--')
        plt.savefig(folder + "-Roc-curve")
        plt.legend(loc='lower right');
        plt.show()
        

In [10]:
from sklearn.metrics import precision_recall_curve, f1_score

def find_best_threshold(data):
    # Compute precision, recall, and thresholds
    precision, recall, thresholds = precision_recall_curve(np.array(data["y-value"].to_list()), data["y-predict"].to_list())

    # Define beta value (β = 2, giving more weight to recall)
    beta = 0.5
    
    # Compute Fβ-Score for all thresholds
    fbeta_scores = (1 + beta**2) * (precision * recall) / (beta**2 * precision + recall + 1e-10)
    
    # Find the threshold with the maximum Fβ-Score
    best_index = np.argmax(fbeta_scores)
    best_threshold = thresholds[best_index]
    best_fbeta_score = fbeta_scores[best_index]

    # # Filter thresholds based on recall >= 0.04
    # valid_indices = [i for i, r in enumerate(recall) if r >= 0.05 - 1e-7]
    # print(len(valid_indices))
    
    # # Find the index of the maximum precision among valid indices
    # best_index = max(valid_indices, key=lambda i: precision[i])
    
    # # Get the corresponding threshold
    # best_threshold = thresholds[best_index]
    
    return best_threshold

In [11]:
def find_threshold_and_calculate_for(data, folder):
    if is_binary_prediction:
        best_threshold = find_best_threshold(data)
        print(f"Best threshold: {best_threshold}")
    
        ticker_thresholds = pd.DataFrame()
        for sector in data["Orig_Ticker"].unique():
            val = find_best_threshold(data[data["Orig_Ticker"] == sector])
            ticker_thresholds = pd.concat([ticker_thresholds, pd.DataFrame({"Ticker": [sector], "Threshold": [val]})], ignore_index=True)
        ticker_thresholds.to_csv(folder + "_threshold.csv", index=False)
        
        return calculate_for(data, folder, val=best_threshold)
        
    else:
        if use_pre_calc_sigma:
            calculate_for(data, folder, min_threshold=0.01, max_threshold= 2.0)
            calculate_for(data, folder, min_threshold=0.0125, max_threshold= 2.0)
            calculate_for(data, folder, min_threshold=0.015, max_threshold= 2.0)
            calculate_for(data, folder, min_threshold=0.0175, max_threshold= 2.0)
            calculate_for(data, folder, min_threshold=0.02, max_threshold= 2.0)
            calculate_for(data, folder, min_threshold=0.0225, max_threshold= 2.0)
            return calculate_for(data, folder, min_threshold=0.025, max_threshold= 2.0)
        else:
            print(two_sigma_cap_per_ticker)
            return calculate_for(data, folder, two_sigma_cap_per_ticker=two_sigma_cap_per_ticker)
            

In [12]:
def objective(trial):
    
    stock_target_mapping = pd.DataFrame()
    
    # Enable eager execution
    tf.compat.v1.enable_eager_execution()
    print(f"Executing Eagerly?: {tf.executing_eagerly()}")
    
    network_type, seq_len = "cnn", SEQUENCE_SIZE

    underlying_target = trial.suggest_categorical("Underlying_target", target_options)
    if do_stock_by_stock:
        stock = trial.suggest_categorical("Stock", stocks)
        folder = "..//models//" + stock + "//"
        current_stocks = [stock]
        print(f"OPTUNA WILL EXECUTE FOR STOCK: {stock} UNDERLYING_TARGET: {underlying_target} NETWORK:{network_type} SEQ_LEN:{seq_len}")
    else:
        folder = "..//models//COMBINED//"
        current_stocks = stocks
        print(f"OPTUNA WILL EXECUTE FOR STOCKS: {stocks} UNDERLYING_TARGET: {underlying_target} NETWORK:{network_type} SEQ_LEN:{seq_len}")
    folder_extended = folder + underlying_target
    
    from pathlib import Path
    directory = Path(folder)
    directory.mkdir(parents=True, exist_ok=True)
    
    file_name_variable = f"{underlying_target}"
    model_name = f"{file_name_variable}.keras"
    
    # Create the ReduceLROnPlateau callback
    reduce_lr = ReduceLROnPlateau(
        monitor=monitor,     # Metric to monitor
        factor=0.5,             # Factor to reduce the learning rate (new_lr = lr * factor)
        patience=REDUCE_LR_PATIENCE,             # Number of epochs with no improvement before reducing
        min_lr=1e-11,# Minimum learning rate
        mode=mode,
        verbose=1
    )
    
    # EarlyStopping to stop training
    early_stopping = EarlyStopping(
        monitor=monitor,
        patience=STOP_L_PATIENCE,
        restore_best_weights=True,
        mode=mode,
        min_delta=early_stop_delta
    )
    
    # Configure logging
    logging.basicConfig(
        filename= folder + f"training_output_{file_name_variable}.log", 
        level=logging.INFO, 
        format='%(asctime)s - %(message)s'
    )
    logger = logging.getLogger()
    
    # Define a custom callback to log training progress
    class LogCallback(tf.keras.callbacks.Callback):
        def on_epoch_end(self, epoch, logs=None):
            logger.info(f"Epoch {epoch + 1}: {logs}")
    
    log_dir = "logs/fit/" + datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
    tensorboard_callback = tf.keras.callbacks.TensorBoard(log_dir=log_dir, histogram_freq=1)
    
    
    train_data, validation_data, test_data, two_sigma_cap_per_ticker, real_target = construct_data(seq_len, current_stocks, underlying_target)
    
    if is_binary_prediction and do_stock_by_stock:
        binary_target_val = validation_data.iloc[-1]['Target-20D-Mean'] + validation_data.iloc[-1]['Target-20D-Sigma']
        stock_target_mapping = pd.concat([stock_target_mapping, pd.DataFrame([{'Ticker': stock, 'Target': underlying_target, 'Target-Val': binary_target_val}])])
        stock_target_mapping.to_csv(folder + underlying_target + "_binary_target_val.csv", index=False)
    
    optimizer = Adam(learning_rate=STARTING_LR)
    if is_binary_prediction:
        count_1 = (train_data['y-value-original'] == 1.0).sum()
        count_0 = (train_data['y-value-original'] == 0.0).sum()
        ratio = count_0 / count_1
        print(f"RATIO::: of y-values = 0 vs 1: {ratio}")
        loss, loss_name = calculate_loss(ratio)
    else:
        loss, loss_name = calculate_loss()
    
    
    # Display rows with NaN values
    nan_rows = train_data[train_data.isnull().any(axis=1)]
    print("Rows with NaN values:")
    print(nan_rows)
    
    # Find specific columns with NaN
    nan_cols = train_data.columns[train_data.isnull().any()].tolist()
    print("Columns with NaN values:", nan_cols)
    
    
    
    if is_binary_prediction:
        # # if NORMALIZE_Y is False:
        # class_weights = compute_class_weight(
        #     class_weight='balanced',
        #     classes=train_data["y-value"].unique(),
        #     y=train_data["y-value"].to_list()
        # )
        # class_weights = dict(enumerate(class_weights))
        class_weights = None
    else:
        class_weights = None
    
    ### Let's convert all of our stuff to np arrays to avoid any indexing issues:
    X_train = np.array(train_data["LstmData"].to_list())
    X_train_ticker = np.array(train_data["Ticker"].to_list())
    X_train_sector = np.array(train_data["Sector"].to_list())
    y_train = np.array(train_data["y-value"].to_list())
    
    X_validate = np.array(validation_data["LstmData"].to_list())
    X_validate_ticker = np.array(validation_data["Ticker"].to_list())
    X_validate_sector = np.array(validation_data["Sector"].to_list())
    y_validate = np.array(validation_data["y-value"].to_list())
    
    nan_rows = np.where(np.isnan(X_train).any(axis=1))[0]  # Get indices of rows with NaN values
    if nan_rows.any():
        nan_rows_tickers = train_data.iloc[nan_rows]
        print(f"{nan_rows_tickers['Orig_Ticker']}")
    
    assert not np.any(np.isnan(X_train)), "Train Data contains NaN values!"
    assert not np.any(np.isinf(X_train)), "Train Data contains infinite values!"
    
    assert not np.any(np.isnan(X_validate)), "Validate Data contains NaN values!"
    assert not np.any(np.isinf(X_validate)), "Validate Data contains infinite values!"
    
    
    
    print(f"\n\n PRINTING NEW STUFF \n\n")
    print(train_data.iloc[0][lstm_features].shape)
    print(train_data.iloc[0][lstm_features])
    print(train_data.iloc[0]["LstmData"])
    print(f"\n\n END OF NEW STUFF \n\n")
    
    
    network = create_model(train_data, network_type, seq_len)
    network.compile(
        optimizer=optimizer,
        loss=loss,
        metrics=[
            precision_with_threshold(0.7),  # Custom precision
            recall_with_threshold(0.7),    # Custom recall
            'AUC',                         # Built-in AUC
            'Precision',                   # Built-in Precision
            'Recall',                      # Built-in Recall
            f1_score_metric,                # Custom F1 score
            true_positives,
            all_positives,
            recall_mul_prediction
        ]
    )
    
    # model_name = f"model_{file_name_variable}-trial-{trial.number}-type-{network_type}-seq-len-{seq_len}.keras"
    best_model_checkpoint_callback = ModelCheckpoint(
        folder + model_name, 
        monitor=save_best_metric, 
        save_best_only=True, 
        mode=save_best_mode, 
        verbose=0)
    
    tf.keras.config.disable_traceback_filtering()
    # Training the model
    history = network.fit(
        [X_train, X_train_ticker, X_train_sector, train_data[lstm_features].to_numpy()]
        ,y_train
        ,validation_data=([X_validate, X_validate_ticker, X_validate_sector, validation_data[lstm_features].to_numpy()], y_validate)
        ,epochs=NUM_EPOCH
        ,batch_size = BATCH_SIZE
        ,callbacks = [best_model_checkpoint_callback, reduce_lr, early_stopping, LogCallback(), tensorboard_callback]
        ,class_weight=class_weights
    )
    
    network.load_weights(folder + model_name)
    
    y_validate_predict = network.predict([X_validate, X_validate_ticker, X_validate_sector, validation_data[lstm_features].to_numpy()])
    validation_data['y-predict'] = y_validate_predict
    for ticker in validation_data["Orig_Ticker"].unique():    
        validation_data.loc[validation_data["Orig_Ticker"] == ticker, 'y-predict-original'] = inverse_normalize_data(validation_data[['y-predict']], real_target, ticker)
    
    
    print(f"\n target: {real_target} \n\n\n")
    print(f"\n\n\n y-predict for validatiion: {y_validate_predict} \n\n\n")
    
    
    y_train_predict = network.predict([X_train, X_train_ticker, X_train_sector, train_data[lstm_features].to_numpy()])
    train_data['y-predict'] = y_train_predict
    for ticker in train_data["Orig_Ticker"].unique():    
        train_data.loc[train_data["Orig_Ticker"] == ticker, 'y-predict-original'] = inverse_normalize_data(train_data[['y-predict']], real_target, ticker)

    if do_stock_by_stock:
        label = stock
    else:
        label = "All stocks"
    
    print_and_save_for(train_data, "Train", folder_extended, label=label)
    print_and_save_for(validation_data, "Validation", folder_extended, label=label)
    
    plot_roc_and_PR_curvers_for(train_data, validation_data, folder_extended)
    return find_threshold_and_calculate_for(validation_data, folder_extended)

In [ ]:
import itertools

study = optuna.create_study(direction="maximize")

if do_stock_by_stock:
    # Run trials for every possible combination
    for ss, uu in itertools.product(stocks, target_options):
        study.enqueue_trial({"Stock": ss, "Underlying_target": uu})
    
    study.optimize(objective, n_trials=len(stocks) * len(target_options))
else:
    # Run trials for every possible combination
    for uu in target_options:
        study.enqueue_trial({"Underlying_target": uu})
    
    study.optimize(objective, len(target_options))

trial = study.best_trial

print("Accuracy: {}".format(trial.value))
print("Best hyperparameters: {}".format(trial.params))

[I 2025-01-19 14:26:47,134] A new study created in memory with name: no-name-366ab180-ec8e-45e7-98fc-eb5f8ee6d9dd
/Users/rifatordulu/Developer/lstm-stock-price-prediction/helpers/data_manipulator.py:133: FutureWarning: In a future version of pandas, parsing datetimes with mixed time zones will raise an error unless `utc=True`. Please specify `utc=True` to opt in to the new behaviour and silence this warning. To create a `Series` with mixed offsets and `object` dtype, please use `apply` and `datetime.datetime.strptime`
  data["Date"] = pd.to_datetime(data["Date"], errors="raise")
/Users/rifatordulu/python_3_11/lib/python3.11/site-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but MinMaxScaler was fitted without feature names
  warnings.warn(
/Users/rifatordulu/python_3_11/lib/python3.11/site-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but MinMaxScaler was fitted without feature names
  warnings.warn(
/Users/rifatordulu/pytho

Executing Eagerly?: True
OPTUNA WILL EXECUTE FOR STOCK: SPOT UNDERLYING_TARGET: Next-Day-High-To-Next-Day-Open-Ratio NETWORK:cnn SEQ_LEN:65
fetch_finance_data_for_tickers: just returning file name: SPOT for interval: 5y
object
{'SPOT': 0}
{'Entertainment': 0}
0      0.0
1      0.0
2      1.0
3      0.0
4      0.0
      ... 
924    0.0
925    0.0
926    0.0
927    0.0
928    0.0
Name: y-value, Length: 929, dtype: float64
0      0.0
1      0.0
2      1.0
3      0.0
4      0.0
      ... 
924    0.0
925    0.0
926    0.0
927    0.0
928    0.0
Name: y-value-original, Length: 929, dtype: float64
High-Low-Ratio            6.243195e-01
High-Open-Ratio           5.707195e-01
Close-Open-Ratio          5.174974e-01
Close-Prev-Close          4.988507e-01
MACD-Prev-MACD            3.697856e-01
200D-Ratio                5.540116e-01
MACD_SIG                  8.052965e-01
MFI3                      3.783944e-01
RSI                       5.735358e-01
RSI3                      5.645773e-01
POCR         

/Users/rifatordulu/Developer/lstm-stock-price-prediction/helpers/data_manipulator.py:496: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  data_frame["Ticker"] = data_frame["Ticker"].replace(stock_dict)
/Users/rifatordulu/Developer/lstm-stock-price-prediction/helpers/data_manipulator.py:511: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  data_frame["Sector"] = data_frame["Sector"].replace(sector_dict)


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ lstm-values         │ (None, 65, 24)    │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d (Conv1D)     │ (None, 63, 256)   │     18,688 │ lstm-values[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ p_re_lu (PReLU)     │ (None, 63, 256)   │        256 │ conv1d[0][0]      │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_1 (Conv1D)   │ (None, 31, 256)   │    196,864 │ p_re_lu[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ p_re_lu_1 (PReLU)   │ (None, 31, 256)   │        256 │ conv1d_1[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_2 (Conv1D)   │ (None, 15, 512)   │    393,728 │ p_re_lu_1[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ p_re_lu_2 (PReLU)   │ (None, 15, 512)   │        512 │ conv1d_2[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_3 (Conv1D)   │ (None, 7, 512)    │    786,944 │ p_re_lu_2[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ p_re_lu_3 (PReLU)   │ (None, 7, 512)    │        512 │ conv1d_3[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_4 (Conv1D)   │ (None, 3, 1024)   │  1,573,888 │ p_re_lu_3[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ p_re_lu_4 (PReLU)   │ (None, 3, 1024)   │      1,024 │ conv1d_4[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stock_id            │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_5 (Conv1D)   │ (None, 1, 1024)   │  3,146,752 │ p_re_lu_4[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stock_embedding     │ (None, 1, 512)    │        512 │ stock_id[0][0]    │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ sector_id           │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ p_re_lu_5 (PReLU)   │ (None, 1, 1024)   │      1,024 │ conv1d_5[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ reshape (Reshape)   │ (None, 512)       │          0 │ stock_embedding[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ sector_embedding    │ (None, 1, 128)    │        128 │ sector_id[0][0]   │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ flatten (Flatten)   │ (None, 1024)      │          0 │ p_re_lu_5[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stock_dense (Dense) │ (None, 512)       │    262,656 │ reshape[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ reshape_1 (Reshape) │ (None, 128)       │          0 │ sector_embedding… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate         │ (None, 1536)      │          0 │ flatten[0][0],    │
│ (Concatenate)       │                   │            │ stock_dense[0][0

 Total params: 10,843,372 (41.36 MB)

 Trainable params: 10,843,372 (41.36 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/200
8/8 ━━━━━━━━━━━━━━━━━━━━ 7s 530ms/step - AUC: 0.5307 - Precision: 0.2220 - Recall: 0.5770 - all_positives: 24.8003 - f1_score_metric: 34.0112 - loss: 29.4045 - precision_with_threshold_fixed: 0.0000e+00 - recall_mul_prediction: 1766.9807 - recall_with_threshold_fixed: 0.0000e+00 - true_positives: 1766.9807 - val_AUC: 0.3622 - val_Precision: 0.1111 - val_Recall: 1.0000 - val_all_positives: 7.0000 - val_f1_score_metric: 12.6000 - val_loss: 27.9708 - val_precision_with_threshold_fixed: 0.0000e+00 - val_recall_mul_prediction: 441.0001 - val_recall_with_threshold_fixed: 0.0000e+00 - val_true_positives: 441.0000 - learning_rate: 1.0000e-04
Epoch 2/200
8/8 ━━━━━━━━━━━━━━━━━━━━ 3s 387ms/step - AUC: 0.6029 - Precision: 0.2213 - Recall: 0.9976 - all_positives: 27.0262 - f1_score_metric: 44.1914 - loss: 27.7853 - precision_with_threshold_fixed: 14.4881 - recall_mul_prediction: 3337.6807 - recall_with_threshold_fixed: 0.9127 - true_positives: 3337.6802 - val_AUC: 0.4133 - val_Precision

In [ ]:
study.storage = optuna.storages.RDBStorage("sqlite:///optuna-results.db")

#### Performance Evaluation

### LET'S TRY THIS ON THE TEST DATA NOW

### SANITATION: